# Reproduction Notebook — arXiv Preprint

**Paper:** Recall-Optimised Failure Detection in Industrial Telemetry:  
A Rolling Degraded-State Counter vs a 21-Sensor Random Forest on NASA C-MAPSS FD001

**Author:** Puru Pandey  
**Repo:** https://github.com/Puru2001pandey/industrial-telemetry-analytics-research

---

## Prerequisites

Place `train_FD001.txt` in `data/train_FD001.txt`.  
The notebook raises `FileNotFoundError` if the file is absent.

**Canonical preprocessing:**
- LR uses per-unit min-max normalisation → health-window feature (1 feature)
- RF uses `StandardScaler` (global) on all 21 raw sensor columns

This matches the original experiment that produced the paper's Table 1 numbers.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, recall_score, precision_score, f1_score, confusion_matrix
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = 'data/train_FD001.txt'
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Data file not found: {DATA_PATH}\n'
        'Download train_FD001.txt from NASA C-MAPSS and place it in data/'
    )
print(f'Data file found: {DATA_PATH}')

## 1. Load Data

In [ ]:
col_names = ['unit_id','cycle','op1','op2','op3'] + [f's{i:02d}' for i in range(1,22)]
df = pd.read_csv(DATA_PATH, sep=r'\s+', header=None, engine='python')
df = df.iloc[:, :len(col_names)]
df.columns = col_names

SENSOR_COLS = [f's{i:02d}' for i in range(1, 22)]

max_cycle = df.groupby('unit_id')['cycle'].max()
df['max_cycle'] = df['unit_id'].map(max_cycle)
df['RUL']       = df['max_cycle'] - df['cycle']
df['at_risk']   = (df['RUL'] <= 30).astype(int)
df = df.sort_values(['unit_id','cycle']).reset_index(drop=True)

print(f'Dataset: {df.shape[0]:,} rows | {df["unit_id"].nunique()} engines')
print(f'At-risk: {df["at_risk"].sum():,} ({df["at_risk"].mean()*100:.1f}%)')

## 2. Health-Window Feature (LR input)

Per-unit min-max normalisation → composite degradation score → rolling 10-cycle degraded-state count.

In [ ]:
KEY_UP   = ['s02','s03','s04','s07','s08','s09','s11','s12','s13','s14','s15']
KEY_DOWN = ['s17','s20','s21']

for col in KEY_UP + KEY_DOWN:
    mn = df.groupby('unit_id')[col].transform('min')
    mx = df.groupby('unit_id')[col].transform('max')
    df[col + '_n'] = (df[col] - mn) / (mx - mn + 1e-9)

up_norm   = [c + '_n' for c in KEY_UP]
down_norm = [c + '_n' for c in KEY_DOWN]
df['deg_score'] = (df[up_norm].mean(axis=1) + (1 - df[down_norm].mean(axis=1))) / 2.0

# NOTE: Q_0.70 computed over full engine trajectory (post-mortem analysis).
# In live deployment, estimate threshold from healthy-phase data only.
p70 = df.groupby('unit_id')['deg_score'].transform(lambda x: x.quantile(0.70))
df['degraded'] = (df['deg_score'] >= p70).astype(int)
df['hw'] = df.groupby('unit_id')['degraded'].transform(
    lambda x: x.rolling(10, min_periods=1).sum())

print(f'Health-window feature: range [0, {int(df["hw"].max())}]')

## 3. Unit-Level 80/20 Split (seed=42, no temporal leakage)

In [ ]:
np.random.seed(42)
units   = df['unit_id'].unique().copy()
np.random.shuffle(units)
split   = int(len(units) * 0.8)
train_u = set(units[:split])
test_u  = set(units[split:])

tr_mask = df['unit_id'].isin(train_u)
te_mask = df['unit_id'].isin(test_u)
y_tr = df.loc[tr_mask, 'at_risk'].values
y_te = df.loc[te_mask, 'at_risk'].values

print(f'Train: {len(train_u)} engines, {tr_mask.sum():,} cycles')
print(f'Test:  {len(test_u)} engines, {te_mask.sum():,} cycles')

## 4. Model A — Logistic Regression (health-window, 1 feature)

In [ ]:
X_lr_tr = df.loc[tr_mask, ['hw']].values
X_lr_te = df.loc[te_mask, ['hw']].values

lr = LogisticRegression(random_state=42, max_iter=2000, class_weight='balanced')
lr.fit(X_lr_tr, y_tr)
yp_lr = lr.predict(X_lr_te)

print('=== Logistic Regression ===')
print(classification_report(y_te, yp_lr, target_names=['Healthy','At-Risk'], digits=4))

## 5. Model B — Random Forest (21 raw sensors, StandardScaler)

**Note:** RF uses global `StandardScaler` on raw sensor columns — *not* per-unit normalisation. This is the preprocessing used in Table 1 of the paper.

In [ ]:
scaler  = StandardScaler()
X_rf_tr = scaler.fit_transform(df.loc[tr_mask, SENSOR_COLS])
X_rf_te = scaler.transform(df.loc[te_mask, SENSOR_COLS])

rf = RandomForestClassifier(n_estimators=100, random_state=42,
                             class_weight='balanced', n_jobs=-1)
rf.fit(X_rf_tr, y_tr)
yp_rf = rf.predict(X_rf_te)

print('=== Random Forest (21 raw sensors, StandardScaler) ===')
print(classification_report(y_te, yp_rf, target_names=['Healthy','At-Risk'], digits=4))

## 6. Summary Table (Table 1 in paper) + Confusion Matrices

In [ ]:
lr_rec = recall_score(y_te, yp_lr, pos_label=1)
rf_rec = recall_score(y_te, yp_rf, pos_label=1)
lr_f1  = f1_score(y_te, yp_lr, average='weighted')
rf_f1  = f1_score(y_te, yp_rf, average='weighted')

print(f'{'Model':<40} {'At-Risk Recall':>15} {'Weighted F1':>13}')
print('-'*70)
print(f'{'LR -- health-window (1 feature)':<40} {lr_rec:>15.4f} {lr_f1:>13.4f}')
print(f'{'RF -- 21 raw sensors':<40} {rf_rec:>15.4f} {rf_f1:>13.4f}')

from sklearn.metrics import confusion_matrix
cm_lr = confusion_matrix(y_te, yp_lr)
cm_rf = confusion_matrix(y_te, yp_rf)
print(f'\nLR confusion matrix: TN={cm_lr[0,0]} FP={cm_lr[0,1]} FN={cm_lr[1,0]} TP={cm_lr[1,1]}')
print(f'RF confusion matrix: TN={cm_rf[0,0]} FP={cm_rf[0,1]} FN={cm_rf[1,0]} TP={cm_rf[1,1]}')

## 7. Statistical Stability — 30 Random Seeds

In [ ]:
print('Running 30-seed stability test...')
lr_recs, rf_recs = [], []

for seed in range(30):
    rng = np.random.RandomState(seed)
    sh  = rng.permutation(units)
    tr_u, te_u = set(sh[:80]), set(sh[80:])
    tr = df[df['unit_id'].isin(tr_u)]
    te = df[df['unit_id'].isin(te_u)]

    m_lr = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
    m_lr.fit(tr[['hw']], tr['at_risk'])
    lr_recs.append(recall_score(te['at_risk'], m_lr.predict(te[['hw']]), pos_label=1))

    sc2 = StandardScaler()
    m_rf = RandomForestClassifier(100, class_weight='balanced', random_state=42, n_jobs=-1)
    m_rf.fit(sc2.fit_transform(tr[SENSOR_COLS]), tr['at_risk'])
    rf_recs.append(recall_score(te['at_risk'], m_rf.predict(sc2.transform(te[SENSOR_COLS])), pos_label=1))

lr_arr, rf_arr = np.array(lr_recs), np.array(rf_recs)
t, p = stats.ttest_rel(lr_arr, rf_arr)
print(f'LR: {lr_arr.mean():.4f} +/- {lr_arr.std():.4f}')
print(f'RF: {rf_arr.mean():.4f} +/- {rf_arr.std():.4f}')
print(f'Paired t-test: t={t:.4f}, p={p:.6f}')
print(f'LR > RF in {(lr_arr > rf_arr).sum()} / 30 splits')